# 三方 AI 辩论（Three-way AI Debate）

## 练习目标（理念）

让三位大语言模型（LLM）围绕同一主题做**结构化辩论**，每人有独立人设与模型后端：

- **Alex（GPT-4.1-mini）** — 主持人：介绍主题、追问、把讨论拉回主线
- **John（Gemini via OpenRouter）** — 乐观派：看机会、谈落地影响
- **Peter（Claude via OpenRouter）** — 怀疑派：质疑假设、要证据

每位模型都会收到：定义角色的 **system prompt**，以及带**完整迄今 transcript** 的 user prompt。

## 和本课的关系

| 概念 | 本练习 |
|------|--------|
| 多客户端 | `openai_client` + `openrouter_client` |
| System / User messages | 人设在 system，上下文在 user |
| 共享状态 | 全局 `transcript` 列表 |

## 怎么跑

1. `.env` 需有可用的 OpenAI 密钥（默认客户端）以及 `OPENROUTER_API_KEY`
2. 可在主题单元格改写 `topic`
3. 依次运行「开场 → 多轮 → 闭幕 → 打印全文」


In [ ]:
# ========== 导入：展示 + OpenAI SDK + 环境变量 ==========

# 从 IPython.display 导入：在笔记本里渲染 Markdown 辩论发言
from IPython.display import display, Markdown
# 从 openai 导入 OpenAI：云端 GPT 与 OpenRouter 共用 SDK
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# 导入标准库 os：用 getenv 读取 OPENROUTER_API_KEY
import os


In [ ]:
# ========== 加载 .env ==========
# override=True：用文件里的值覆盖进程中已有同名环境变量

load_dotenv(override=True)


### 客户端（Clients）

两个 OpenAI 兼容客户端：官方 OpenAI（Alex）与 OpenRouter（John / Peter）。


In [ ]:
# ========== 创建两个 API 客户端 ==========

# 默认 OpenAI 客户端：密钥来自环境变量 OPENAI_API_KEY（SDK 约定）
openai_client = OpenAI()

# OpenRouter 客户端：base_url / api_key 字符串保持原样
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)


### 模型与人物（Models & Personas）

常量集中存放模型 id；改这里即可换模型，勿改调用逻辑。


In [ ]:
# ========== 三角色各自绑定的模型 id ==========
# 字符串必须与账号可用模型一致；勿翻译或「美化」id

ALEX_MODEL = "gpt-4.1-mini"
JOHN_MODEL = "google/gemini-2.5-flash-lite"
PETER_MODEL = "anthropic/claude-3.5-haiku"


In [ ]:
# ========== System prompts：三人设（发给模型的英文正文保持原样） ==========
# 教学提示：system 定性格与发言长度；改译会改变辩论风格

# Alex：主持人——追问、挑战弱论证、控制篇幅
alex_system = """You are Alex, the moderator of a debate.
You introduce the topic, ask probing follow-up questions, challenge weak arguments,
and keep the discussion focused. You are fair but push both sides to go deeper.
Keep your responses concise — 2 to 4 sentences max.
You are in a debate with John and Peter."""

# John：乐观、前瞻，举例谈现实影响
john_system = """You are John, a debater who is optimistic and forward-thinking.
You see opportunity and potential in new developments. You back your points
with practical examples and real-world impact.
Keep your responses concise — 3 to 5 sentences max.
You are in a debate with Peter. Alex is the moderator."""

# Peter：怀疑、分析，压力测试想法（不是无脑否定）
peter_system = """You are Peter, a debater who is skeptical and analytical.
You question assumptions, point out risks, and demand evidence.
You're not negative — you just want to stress-test ideas before buying in.
Keep your responses concise — 3 to 5 sentences max.
You are in a debate with John. Alex is the moderator."""


### 主题（Topic）

改下一格的 `topic` 字符串即可辩论别的问题。


In [ ]:
# ========== 辩论主题（改这里即可换题） ==========
# 发给模型的英文主题句保持原样

topic = "Why does it matter right now to learn LLM engineering? Is this the right time, or is it too early / too late?"


In [ ]:
# ========== 共享 transcript：单一事实来源 ==========
# 每位发言后 append；call_* 都会读它拼进 user prompt

transcript = []


In [ ]:
# ========== 把 transcript 格式化成可读字符串 ==========
# 供塞进各角色的 user_prompt；空列表时返回占位句


def format_transcript():
    # 尚无发言：返回占位（英文保持原样，模型会读到）
    if not transcript:
        return "(No conversation yet)"
    lines = []
    # 每条记录是 {"speaker": ..., "text": ...}
    for entry in transcript:
        lines.append(f"{entry['speaker']}: {entry['text']}")
    # 发言之间空一行，便于模型分段阅读
    return "\n\n".join(lines)


In [ ]:
# ========== call_alex：主持人发言并写入 transcript ==========
# instruction：本轮具体任务（开场 / 追问 / 总结）；上下文来自 format_transcript()


def call_alex(instruction):
    # user_prompt：指令 + 迄今对话 +「以 Alex 身份回答」（英文模板保持原样）
    user_prompt = f"""{instruction}

The conversation so far:
{format_transcript()} 

Respond as Alex the moderator."""
    
    # 走官方 OpenAI 客户端 + ALEX_MODEL
    response = openai_client.chat.completions.create(
        model=ALEX_MODEL,
        messages=[
            {"role": "system", "content": alex_system},
            {"role": "user", "content": user_prompt}
        ],
    )

    # 取出助手文本，追加到共享记录后返回
    reply = response.choices[0].message.content
    transcript.append({"speaker": "Alex (Moderator)", "text": reply})
    return reply


In [ ]:
# ========== call_john：乐观派发言并写入 transcript ==========


def call_john():
    # user_prompt 固定模板：提醒身份 + 对话历史 + 保持乐观人设（英文保持原样）
    user_prompt = f"""You are John in a debate moderated by Alex.
The conversation so far:
{format_transcript()}
Respond to what just been said, Stay in character as the optimistic and forward-thinking John."""
    
    # 经 OpenRouter 调用 Gemini 模型 id
    response = openrouter_client.chat.completions.create(
        model=JOHN_MODEL,
        messages=[
            {"role": "system", "content": john_system},
            {"role": "user", "content": user_prompt}
        ],
    )
    reply = response.choices[0].message.content
    transcript.append({"speaker": "John (Optimist)", "text": reply})
    return reply


In [ ]:
# ========== call_peter：怀疑派发言并写入 transcript ==========


def call_peter():
    # user_prompt：身份 + 历史 + 保持怀疑人设（英文保持原样）
    user_prompt = f"""You are Peter in a debate moderated by Alex.

The conversation so far:
{format_transcript()}

Respond to what was just said. Stay in character as the skeptic."""

    # 经 OpenRouter 调用 Claude 模型 id
    response = openrouter_client.chat.completions.create(
        model=PETER_MODEL,
        messages=[
            {"role": "system", "content": peter_system},
            {"role": "user", "content": user_prompt}
        ]
    )
    reply = response.choices[0].message.content
    transcript.append({"speaker": "Peter (Skeptic)", "text": reply})
    return reply


In [ ]:
# ========== show：在笔记本里漂亮展示一位发言 ==========


def show(speaker, text):
    # 用 Markdown 三级标题 + 正文 + 分隔线
    display(Markdown(f"### {speaker}\n\n{text}\n\n---"))


### 辩论主循环（Debate）

先重置 transcript 并开场，再多轮 John → Peter → Alex，最后闭幕。


In [ ]:
# ========== 第 0 轮：Alex 介绍主题 ==========

# 重置共享记录，避免重复跑单元格时叠加上次辩论
transcript = []  # reset

# 把 topic 嵌进开场指令（英文指令字符串保持原样）
opening = call_alex(f"Introduce this debate topic to John and Peter: {topic}")
# 在笔记本中展示主持人开场
show("Alex (Moderator)", opening)


In [ ]:
# ========== 第 1–5 轮：John → Peter → Alex 追问 ==========

for i in range(1, 6):
    # 轮次标题（Round 文案保持原样）
    display(Markdown(f"## Round {i}"))

    # 约翰回应（乐观派）
    john_reply = call_john()
    show("John (Optimist)", john_reply)

    # 彼得回应（怀疑派）
    peter_reply = call_peter()
    show("Peter (Skeptic)", peter_reply)

    # 亚历克斯追问/挑战（英文 instruction 保持原样）
    alex_reply = call_alex("Ask a follow-up question or challenge one of them. Push the debate deeper.")
    show("Alex (Moderator)", alex_reply)


In [ ]:
# ========== 闭幕：Alex 总结双方要点 ==========
# 英文总结指令保持原样

closing = call_alex("Summarize the key points from both sides and give your closing remarks to end the debate.")
show("Alex (Moderator)", closing)


### 完整成绩单（Full Transcript）

打印 `format_transcript()`，便于复制存档。


In [ ]:
# ========== 打印完整辩论记录 ==========

print(format_transcript())
